In [ ]:
!pip install transformers Levenshtein sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.6 MB/s eta 0:00:00


In [ ]:
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm
from datetime import datetime
import shutil

# ================= CONFIG =================
class CONFIG:
    TRAIN_CSV = "/content/drive/MyDrive/Text Correction/train.csv"
    TEST_CSV = "/content/drive/MyDrive/Text Correction/test.csv"

    # Khởi tạo Run ID và thư mục lưu trên Drive
    BASE_DIR = "/content/drive/MyDrive/Text Correction"
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
    RUN_DIR = os.path.join(BASE_DIR, f"Seq2Seq_run_{RUN_ID}")

    # Tự động backup file notebook
    NOTEBOOK_PATH = "/content/drive/MyDrive/Colab Notebooks/Viet Text Correction"


os.makedirs(CONFIG.RUN_DIR, exist_ok=True)
print(f"Lưu kết quả chạy (run) tại: {CONFIG.RUN_DIR}")

if os.path.exists(CONFIG.NOTEBOOK_PATH):
    shutil.copy(CONFIG.NOTEBOOK_PATH, os.path.join(CONFIG.RUN_DIR, "notebook_backup.ipynb"))
    print("Đã tự động sao lưu file notebook vào thư mục run!")



Lưu kết quả chạy (run) tại: /content/drive/MyDrive/Text Correction/Seq2Seq_run_20260531_063526
Đã tự động sao lưu file notebook vào thư mục run!


In [ ]:
import pandas as pd
import re
import os

def create_sample_submission():
    print("Đang tạo sample submission...")
    # 1. Đọc test.csv
    df_test = pd.read_csv(CONFIG.TEST_CSV)

    # Helper function để thay thế các ký tự lỗi UTF-8 bằng khoảng trắng
    def clean_utf8_errors(text):
        if not isinstance(text, str): # Đảm bảo đầu vào là string
            return ""
        # Mã hóa sang utf-8, bỏ qua các ký tự không hợp lệ, sau đó giải mã lại
        return text.encode('utf-8', 'ignore').decode('utf-8').strip()

    # 2. Quét qua cột 'input', tìm và thay thế ký tự lỗi UTF-8
    # Sử dụng .copy() để tránh SettingWithCopyWarning
    df_cleaned = df_test.copy()
    df_cleaned['corrected_text'] = df_cleaned['input'].apply(clean_utf8_errors)

    # 3. Tạo DataFrame gồm id và corrected_text
    submission_df = df_cleaned[['id', 'corrected_text']]

    # 4. Lưu vào working directory
    submission_path = os.path.join(CONFIG.RUN_DIR, "sample_submission.csv")
    submission_df.to_csv(submission_path, index=False)

    print(f"Sample submission đã được lưu tại: {submission_path}")

# Gọi hàm để thực thi
create_sample_submission()

Đang tạo sample submission...
Sample submission đã được lưu tại: /content/drive/MyDrive/Text Correction/Seq2Seq_run_20260531_063526/sample_submission.csv


### 🌟 Thuật toán Statistical Machine Learning (Word-Level MLE)

Thuật toán này học một mô hình ngôn ngữ dịch thuật thống kê cơ bản.
1. **Gióng hàng (Alignment):** Lấy các câu có cùng số lượng từ giữa `input` và `corrected_text` để gióng hàng 1-1.
2. **Đếm tần suất (Statistics):** Tính phân phối xác suất $P(Corrected\_Word | Input\_Word)$.
3. **Suy luận (Inference):** Dùng kỹ thuật Maximum Likelihood Estimation (MLE) để chọn ra từ thay thế có tần suất cao nhất, nếu từ (Out-Of-Vocabulary) không có trong tập thống kê thì giữ nguyên từ gốc (Backoff).

In [ ]:
import pandas as pd
import os
from collections import defaultdict, Counter
from tqdm import tqdm
import Levenshtein

def run_statistical_ml_correction():
    print("1. Đang huấn luyện mô hình Statistical ML (Word-level MLE)...")
    # Đọc dữ liệu Train
    df_train = pd.read_csv(CONFIG.TRAIN_CSV).dropna()

    # Ánh xạ đếm tần suất: P(Corrected_Word | Input_Word)
    translation_counts = defaultdict(Counter)

    # Trích xuất dữ liệu thống kê từ các câu có cùng số lượng từ
    # (để gióng hàng 1-1 chính xác, đảm bảo chất lượng từ điển thống kê)
    for _, row in tqdm(df_train.iterrows(), total=len(df_train), desc="Extracting Statistics"):
        in_words = str(row['input']).split()
        out_words = str(row['corrected_text']).split()

        # Gióng hàng 1-1
        if len(in_words) == len(out_words):
            for iw, ow in zip(in_words, out_words):
                translation_counts[iw][ow] += 1

    # Lấy bản dịch có xác suất cao nhất (Maximum Likelihood)
    best_translation = {}
    for iw, ow_counts in translation_counts.items():
        # most_common(1) trả về list có dạng [(word, count)]
        best_translation[iw] = ow_counts.most_common(1)[0][0]

    print(f"\n=> Đã học được {len(best_translation)} quy tắc sửa từ thống kê.")

    print("\n1.5. Đang đánh giá trên tập Train...")
    total_cer_dist = 0
    total_cer_chars = 0
    for _, row in tqdm(df_train.iterrows(), total=len(df_train), desc="Eval Train"):
        in_words = str(row['input']).split()
        gt_text = str(row['corrected_text'])

        corr_words = [best_translation.get(w, w) for w in in_words]
        pred_text = " ".join(corr_words)

        total_cer_dist += Levenshtein.distance(pred_text, gt_text)
        total_cer_chars += len(gt_text)

    cer = total_cer_dist / total_cer_chars if total_cer_chars > 0 else 0
    print(f"=> Độ lỗi CER (Character Error Rate) trên tập Train: {cer:.4f}")

    print("\n2. Đang dự đoán trên tập Test...")
    df_test = pd.read_csv(CONFIG.TEST_CSV)
    predictions = []

    for text in tqdm(df_test['input'], desc="Predicting"):
        if not isinstance(text, str) or not text.strip():
            predictions.append("")
            continue

        words = text.split()
        # Kỹ thuật Backoff: Nếu từ lỗi có trong từ điển thống kê thì sửa, ngược lại giữ nguyên từ gốc
        corr_words = [best_translation.get(w, w) for w in words]
        predictions.append(" ".join(corr_words))

    # 3. Lưu kết quả Submission
    submission_df = pd.DataFrame({
        'id': df_test['id'],
        'corrected_text': predictions
    })

    out_path = os.path.join(CONFIG.RUN_DIR, "statistical_ml_submission.csv")
    submission_df.to_csv(out_path, index=False)
    print(f"\nHoàn tất! Kết quả thuật toán Statistical ML được lưu tại: {out_path}")

# Chạy thuật toán
run_statistical_ml_correction()


ModuleNotFoundError: No module named 'Levenshtein'

### 🌟 Thuật toán Noisy Channel Model

Sử dụng định lý Bayes để tách biệt mô hình ngôn ngữ (Language Model) và mô hình nhiễu (Error/Channel Model):
- **$P(w)$**: Xác suất tiên nghiệm của từ đúng $w$.
- **$P(x|w)$**: Xác suất từ đúng $w$ bị gõ sai thành từ $x$.

In [ ]:
import os
import math
import pandas as pd
import Levenshtein
from collections import defaultdict, Counter
from tqdm import tqdm

# =======================================================
# LƯU Ý: Giữ nguyên class CONFIG của bạn ở trên đoạn này
# =======================================================

def run_pure_viterbi_noisy_channel():
    print("1. Đang huấn luyện Pure Noisy Channel (Log-Space + Bigram Viterbi)...")
    df_train = pd.read_csv(CONFIG.TRAIN_CSV).dropna()

    # Khởi tạo các biến đếm thống kê
    unigram_counts = Counter()
    bigram_counts = Counter()
    error_counts = defaultdict(Counter)
    x_to_w_candidates = defaultdict(set)

    # --- HỌC TỪ DỮ LIỆU ---
    for _, row in tqdm(df_train.iterrows(), total=len(df_train), desc="Extracting Statistics"):
        in_words = str(row['input']).split()
        out_words = str(row['corrected_text']).split()

        # 1. Train Language Model (Unigram & Bigram)
        for i, ow in enumerate(out_words):
            unigram_counts[ow] += 1
            if i > 0:
                bigram_counts[(out_words[i-1], ow)] += 1

        # 2. Train Error Model P(x|w)
        if len(in_words) == len(out_words):
            for x, w in zip(in_words, out_words):
                error_counts[w][x] += 1
                x_to_w_candidates[x].add(w)

    vocab_size = len(unigram_counts)
    total_words = sum(unigram_counts.values())

    # --- HÀM TÍNH XÁC SUẤT (LOG-SPACE & SMOOTHING) ---
    def get_log_prob_w(prev_w, curr_w):
        """Tính Log(P(curr_w | prev_w)) với Laplace Smoothing."""
        if prev_w is None:
            count = unigram_counts[curr_w]
            return math.log((count + 1) / (total_words + vocab_size))

        count_bi = bigram_counts.get((prev_w, curr_w), 0)
        count_uni = unigram_counts.get(prev_w, 0)
        return math.log((count_bi + 1) / (count_uni + vocab_size))

    def get_log_prob_x_given_w(x, w):
        """Tính Log(P(x|w))."""
        if w in error_counts and x in error_counts[w]:
            return math.log(error_counts[w][x] / unigram_counts[w])

        # Nếu chưa từng thấy lỗi này trong Train, dùng phạt Levenshtein
        dist = Levenshtein.distance(x, w)
        return math.log((0.01) ** dist) if dist > 0 else 0.0

    print("\n2. Đang giải mã tập Test bằng Viterbi...")
    df_test = pd.read_csv(CONFIG.TEST_CSV)
    predictions = []

    for text in tqdm(df_test['input'], desc="Predicting"):
        if not isinstance(text, str) or not text.strip():
            predictions.append("")
            continue

        words = text.split()
        if not words:
            predictions.append("")
            continue

        # --- VITERBI DECODING TÌM ĐƯỜNG ĐI XÁC SUẤT CAO NHẤT ---
        V = [{}]
        path = {}

        # 1. Khởi tạo tập ứng viên (Chỉ lấy những gì đã học từ Train)
        list_candidates = []
        for x in words:
            cands = set(x_to_w_candidates.get(x, []))
            cands.add(x) # Luôn có tùy chọn giữ nguyên từ gốc
            list_candidates.append(cands)

        # 2. Tính điểm cho từ đầu tiên (t = 0)
        for w in list_candidates[0]:
            V[0][w] = get_log_prob_x_given_w(words[0], w) + get_log_prob_w(None, w)
            path[w] = [w]

        # 3. Duyệt Viterbi cho các từ tiếp theo (t > 0)
        for t in range(1, len(words)):
            V.append({})
            new_path = {}
            curr_x = words[t]

            for curr_w in list_candidates[t]:
                max_score = -float('inf')
                best_prev = None

                emission_log_prob = get_log_prob_x_given_w(curr_x, curr_w)

                for prev_w in V[t-1]:
                    transition_log_prob = get_log_prob_w(prev_w, curr_w)
                    score = V[t-1][prev_w] + transition_log_prob + emission_log_prob

                    if score > max_score:
                        max_score = score
                        best_prev = prev_w

                V[t][curr_w] = max_score
                new_path[curr_w] = path[best_prev] + [curr_w]

            path = new_path

        # 4. Chốt chuỗi từ có tổng xác suất cao nhất
        best_final_word = max(V[-1], key=V[-1].get)
        best_sequence = path[best_final_word]

        predictions.append(" ".join(best_sequence))

    # --- LƯU KẾT QUẢ ---
    submission_df = pd.DataFrame({'id': df_test['id'], 'corrected_text': predictions})
    out_path = os.path.join(CONFIG.RUN_DIR, "pure_viterbi_noisy_channel_submission.csv")
    submission_df.to_csv(out_path, index=False)
    print(f"\nHoàn tất! Kết quả thuật toán lõi được lưu tại: {out_path}")

# Chạy thuật toán
run_pure_viterbi_noisy_channel()

1. Đang huấn luyện Pure Noisy Channel (Log-Space + Bigram Viterbi)...


Extracting Statistics: 100%|██████████| 50000/50000 [00:15<00:00, 3325.07it/s]



2. Đang giải mã tập Test bằng Viterbi...


Predicting: 100%|██████████| 10000/10000 [00:56<00:00, 176.37it/s]



Hoàn tất! Kết quả thuật toán lõi được lưu tại: /content/drive/MyDrive/Text Correction/Seq2Seq_run_20260530_052309/pure_viterbi_noisy_channel_submission.csv


In [ ]:
# Mô hình Seq2Seq cũ đã được xóa theo yêu cầu.
